# SOTA-ViT v1 - ViT-Base with Temporal Attention & Adapters (Windows Local)
Fine-tune pretrained ViT-Base on HMDB-51 with SOTA improvements:
- Temporal Attention mechanism
- AdaptFormer-style adapters
- Advanced augmentation (RandAugment, Mixup, Label Smoothing)
- Gradient accumulation, warmup scheduler, early stopping
- Test-Time Augmentation (TTA)

**Notebook này được điều chỉnh để chạy local trên Windows với RTX 5090**


In [ ]:
# Install packages if needed (uncomment if running for the first time)
# !pip install torch torchvision timm tqdm pillow matplotlib numpy

# Fix NumPy compatibility: Downgrade from 2.4.0rc1 to stable 2.3.x
# NumPy 2.4.0rc1 has compatibility issues with matplotlib 3.10.8
import numpy as np
if np.__version__.startswith('2.4'):
    print("⚠️  NumPy 2.4.0rc1 detected - has compatibility issues with matplotlib")
    print("   Downgrading to NumPy 2.3.x for stability...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "numpy<2.4", "--quiet"])
    print("✅ NumPy downgraded. Please restart kernel and run cells again.")
    raise RuntimeError("Please restart kernel after NumPy downgrade")
else:
    print(f"✅ NumPy version {np.__version__} is compatible")


In [ ]:
# Extract HMDB51.zip if data folder doesn't exist
import zipfile
import os
from pathlib import Path

DATA_DIR = './hmdb51_data'
ZIP_FILE = './HMDB51.zip'

if not os.path.exists(DATA_DIR):
    if os.path.exists(ZIP_FILE):
        print(f"Extracting {ZIP_FILE} to {DATA_DIR}...")
        os.makedirs(DATA_DIR, exist_ok=True)
        with zipfile.ZipFile(ZIP_FILE, 'r') as zip_ref:
            zip_ref.extractall(DATA_DIR)
        print("Extraction complete!")
    else:
        print(f"Warning: {ZIP_FILE} not found. Please ensure HMDB51.zip is in the current directory.")
else:
    print(f"Data directory {DATA_DIR} already exists. Skipping extraction.")


In [ ]:
from __future__ import annotations
import math
import random
from pathlib import Path
from typing import Optional, Tuple, Dict, List
import re

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

# Workaround settings cho RTX 5090 (set trước khi dùng GPU)
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
torch.backends.cudnn.benchmark = False  # Disable cuDNN auto-tuning for RTX 5090

# Import torchvision với error handling
try:
    from torchvision import transforms
    from torchvision.transforms import InterpolationMode
    import torchvision.transforms.functional as TF
    TORCHVISION_AVAILABLE = True
except RuntimeError as e:
    if "operator torchvision::nms does not exist" in str(e):
        print("⚠️  Lỗi: torchvision không tương thích với PyTorch")
        print("   Giải pháp: Đóng Jupyter notebook HOÀN TOÀN và chạy lệnh sau trong terminal:")
        print("   conda activate pytorch_gpu")
        print("   pip uninstall -y torchvision")
        print("   pip install torchvision --index-url https://download.pytorch.org/whl/cu128")
        print("   Sau đó mở lại notebook và restart kernel")
        raise
    else:
        raise

from tqdm.auto import tqdm
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import timm

# Use GPU if available (no CPU fallback)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'PyTorch version: {torch.__version__}')

# Check GPU info
if torch.cuda.is_available():
    print(f'CUDA version: {torch.version.cuda}')
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB')
    
    if "5090" in torch.cuda.get_device_name(0):
        if "2.9.1" in torch.__version__:
            print('\n✅ RTX 5090 detected with PyTorch 2.9.1')
            print('   Note: Some CUDA kernels may not be fully supported yet')
        else:
            print('\n⚠️  RTX 5090 detected!')
            print(f'   Current PyTorch: {torch.__version__} may have CUDA compatibility issues')
            print('   Upgrade to PyTorch 2.9.1 with CUDA 12.8 for better RTX 5090 support')
else:
    print('⚠️  No GPU detected - training will be very slow on CPU')


## 1. Configuration


In [ ]:
# Data paths (Windows Local)
DATA_ROOT = './hmdb51_data'

# Model parameters 
NUM_FRAMES = 16
FRAME_STRIDE = 2
IMG_SIZE = 224

# Training parameters - SOTA improvements
BATCH_SIZE = 12  # Reduced for ViT-Base with 16 frames
EPOCHS = 50  # Increased with early stopping
BASE_LR = 5e-5  # Lower LR for backbone
HEAD_LR = 1e-3  # Higher LR for head/adapters
WEIGHT_DECAY = 0.01  # Reduced weight decay
GRAD_ACCUM_STEPS = 4  # Gradient accumulation
VAL_RATIO = 0.15  # 15% validation split
LABEL_SMOOTHING = 0.1
MIXUP_ALPHA = 0.4
WARMUP_EPOCHS = 5
EARLY_STOP_PATIENCE = 10

# Model architecture - Upgraded to ViT-Base
PRETRAINED_NAME = 'vit_base_patch16_224'
USE_ADAPTERS = True  # Use AdaptFormer-style fine-tuning

# Windows: Use 0 workers to avoid multiprocessing issues
NUM_WORKERS = 0

print(f"Data root: {DATA_ROOT}")
print(f"Model: {PRETRAINED_NAME}")
print(f"Frames per video: {NUM_FRAMES}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Epochs: {EPOCHS} (with early stopping)")
print(f"Validation ratio: {VAL_RATIO}")
print(f"Use adapters: {USE_ADAPTERS}")
print(f"Gradient accumulation steps: {GRAD_ACCUM_STEPS}")


## 2. SOTA Model Architecture


In [ ]:
class TemporalAttention(nn.Module):
    """Multi-head temporal attention mechanism for video sequences."""
    
    def __init__(self, embed_dim, num_heads=8, dropout=0.1):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        assert self.head_dim * num_heads == embed_dim, "embed_dim must be divisible by num_heads"
        
        self.scale = self.head_dim ** -0.5
        
        # QKV projections
        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(embed_dim)
        
    def forward(self, x):
        """
        Args:
            x: [B, T, embed_dim] - temporal sequence of features
        Returns:
            out: [B, embed_dim] - aggregated temporal features
        """
        B, T, C = x.shape
        
        # Layer norm
        x_norm = self.norm(x)
        
        # QKV projection
        qkv = self.qkv(x_norm).reshape(B, T, 3, self.num_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # [3, B, num_heads, T, head_dim]
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        # Attention
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.dropout(attn)
        
        # Apply attention to values
        out = (attn @ v).transpose(1, 2).reshape(B, T, C)
        out = self.proj(out)
        out = self.dropout(out)
        
        # Residual connection
        out = out + x
        
        # Global average pooling across temporal dimension
        out = out.mean(dim=1)  # [B, embed_dim]
        
        return out


class Adapter(nn.Module):
    """Lightweight adapter module for AdaptFormer-style fine-tuning."""
    
    def __init__(self, embed_dim, adapter_dim=None, dropout=0.1):
        super().__init__()
        adapter_dim = adapter_dim or (embed_dim // 4)  # Default: 1/4 of embed_dim
        self.adapter_dim = adapter_dim
        
        # Down projection
        self.down_proj = nn.Linear(embed_dim, adapter_dim)
        # Activation
        self.activation = nn.GELU()
        # Up projection
        self.up_proj = nn.Linear(adapter_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        """
        Args:
            x: [B, embed_dim] or [B, T, embed_dim]
        Returns:
            out: same shape as x
        """
        original_shape = x.shape
        if len(original_shape) == 3:
            B, T, C = original_shape
            x = x.view(B * T, C)
        else:
            B, C = original_shape
        
        # Adapter forward
        out = self.down_proj(x)
        out = self.activation(out)
        out = self.dropout(out)
        out = self.up_proj(out)
        
        if len(original_shape) == 3:
            out = out.view(original_shape)
        
        return out


class SOTAViTForAction(nn.Module):
    """SOTA ViT for action recognition with temporal attention and adapters."""
    
    def __init__(self, num_classes=51, pretrained_name='vit_base_patch16_224', 
                 use_adapters=True, adapter_dim=None, temporal_heads=8, dropout=0.1):
        super().__init__()
        
        # Load pretrained ViT-Base
        self.vit = timm.create_model(pretrained_name, pretrained=True, num_classes=0)
        
        # Get embedding dimension
        self.embed_dim = self.vit.num_features
        self.use_adapters = use_adapters
        
        # Freeze backbone if using adapters
        if use_adapters:
            for param in self.vit.parameters():
                param.requires_grad = False
        
        # Temporal attention mechanism
        self.temporal_attention = TemporalAttention(
            embed_dim=self.embed_dim,
            num_heads=temporal_heads,
            dropout=dropout
        )
        
        # Adapters (if enabled)
        if use_adapters:
            self.adapters = nn.ModuleList([
                Adapter(self.embed_dim, adapter_dim, dropout)
                for _ in range(len(self.vit.blocks))
            ])
        
        # Classification head with dropout
        self.head = nn.Sequential(
            nn.LayerNorm(self.embed_dim),
            nn.Dropout(dropout),
            nn.Linear(self.embed_dim, self.embed_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(self.embed_dim // 2, num_classes)
        )
        
    def forward(self, video):
        '''
        Args:
            video: [B, T, C, H, W] - batch of video clips
        Returns:
            logits: [B, num_classes]
        '''
        B, T, C, H, W = video.shape
        
        # Reshape to process all frames
        x = video.view(B * T, C, H, W)
        
        # Extract features with ViT
        # When using adapters, backbone is frozen but we still use it normally
        # Adapters are trained separately and help with fine-tuning
        features = self.vit(x)  # [B*T, embed_dim]
        
        # Reshape back to temporal sequence
        features = features.view(B, T, self.embed_dim)
        
        # Temporal attention
        pooled = self.temporal_attention(features)  # [B, embed_dim]
        
        # Classification
        logits = self.head(pooled)
        
        return logits

print("SOTA ViT with Temporal Attention defined")
print(f"  Backbone: {PRETRAINED_NAME}")
print(f"  Use adapters: {USE_ADAPTERS}")


## 3. Advanced Data Augmentation


In [ ]:
class VideoTransform:
    """Advanced video augmentation with RandAugment, color jitter, and temporal augmentation."""
    
    def __init__(self, image_size=224, is_train=True):
        self.image_size = image_size
        self.is_train = is_train
        # ImageNet normalization stats
        self.mean = [0.485, 0.456, 0.406]
        self.std = [0.229, 0.224, 0.225]
        
        # RandAugment operations
        if is_train:
            try:
                from torchvision.transforms import RandAugment
                self.rand_augment = RandAugment(num_ops=2, magnitude=9)
            except:
                self.rand_augment = None
        else:
            self.rand_augment = None
    
    def _apply_color_jitter(self, frame):
        """Apply color jitter augmentation."""
        if random.random() < 0.5:
            brightness = random.uniform(0.8, 1.2)
            contrast = random.uniform(0.8, 1.2)
            saturation = random.uniform(0.8, 1.2)
            hue = random.uniform(-0.1, 0.1)
            frame = TF.adjust_brightness(frame, brightness)
            frame = TF.adjust_contrast(frame, contrast)
            frame = TF.adjust_saturation(frame, saturation)
            frame = TF.adjust_hue(frame, hue)
        return frame
    
    def _apply_random_erasing(self, frame):
        """Apply random erasing augmentation."""
        if random.random() < 0.3:
            h, w = frame.shape[-2:]
            area = h * w
            erase_area = random.uniform(0.02, 0.33) * area
            aspect_ratio = random.uniform(0.3, 3.3)
            h_erase = int(round((erase_area * aspect_ratio) ** 0.5))
            w_erase = int(round((erase_area / aspect_ratio) ** 0.5))
            if h_erase < h and w_erase < w:
                top = random.randint(0, h - h_erase)
                left = random.randint(0, w - w_erase)
                frame[..., top:top+h_erase, left:left+w_erase] = random.uniform(0, 1)
        return frame
    
    def __call__(self, frames):
        """
        Args:
            frames: [T, C, H, W] tensor of frames
        Returns:
            frames: [T, C, H, W] augmented frames
        """
        T = frames.shape[0]
        
        if self.is_train:
            # Spatial augmentation
            h, w = frames.shape[-2:]
            
            # Random resized crop
            scale = random.uniform(0.8, 1.0)
            new_h, new_w = int(h * scale), int(w * scale)
            frames = TF.resize(frames, [new_h, new_w], interpolation=InterpolationMode.BILINEAR)
            
            # Random crop
            i = random.randint(0, max(0, new_h - self.image_size))
            j = random.randint(0, max(0, new_w - self.image_size))
            frames = TF.crop(frames, i, j, min(self.image_size, new_h), min(self.image_size, new_w))
            frames = TF.resize(frames, [self.image_size, self.image_size], interpolation=InterpolationMode.BILINEAR)
            
            # Horizontal flip
            if random.random() < 0.5:
                frames = TF.hflip(frames)
            
            # Apply augmentations to each frame
            augmented_frames = []
            for t in range(T):
                frame = frames[t]
                
                # RandAugment (if available) - requires uint8 input
                if self.rand_augment is not None and random.random() < 0.5:
                    # Convert from float32 (0-1) to uint8 (0-255) for RandAugment
                    frame_uint8 = (frame * 255).clamp(0, 255).to(torch.uint8)
                    frame_uint8 = self.rand_augment(frame_uint8)
                    # Convert back to float32 (0-1)
                    frame = frame_uint8.to(torch.float32) / 255.0
                
                # Color jitter
                frame = self._apply_color_jitter(frame)
                
                # Random erasing
                frame = self._apply_random_erasing(frame)
                
                augmented_frames.append(frame)
            
            frames = torch.stack(augmented_frames)
        else:
            # Validation: just resize
            frames = TF.resize(frames, [self.image_size, self.image_size], interpolation=InterpolationMode.BILINEAR)
        
        # Normalize with ImageNet stats
        normalized = [TF.normalize(frame, self.mean, self.std) for frame in frames]
        return torch.stack(normalized)


def mixup_data(videos, labels, alpha=0.4):
    """Apply Mixup augmentation to video batch."""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    
    batch_size = videos.size(0)
    index = torch.randperm(batch_size).to(videos.device)
    
    mixed_videos = lam * videos + (1 - lam) * videos[index, :]
    labels_a, labels_b = labels, labels[index]
    return mixed_videos, labels_a, labels_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    """Compute loss for Mixup."""
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

print("Advanced augmentation defined")
print("  Includes: RandAugment, Color Jitter, Random Erasing, Mixup")


## 4. Dataset Classes


In [ ]:
class VideoDataset(Dataset):
    """Video dataset with grouped train/val split to avoid data leakage."""
    
    def __init__(self, root, num_frames=16, frame_stride=2, image_size=224, 
                 is_train=True, val_ratio=0.15, seed=42, samples=None):
        self.root = Path(root)
        self.num_frames = num_frames
        self.frame_stride = frame_stride
        self.transform = VideoTransform(image_size, is_train)
        self.to_tensor = transforms.ToTensor()
        self.classes = sorted([d.name for d in self.root.iterdir() if d.is_dir()])
        self.class_to_idx = {name: idx for idx, name in enumerate(self.classes)}
        
        if samples is None:
            # Collect all samples grouped by video
            grouped_samples = {}  # {(class, base_video_name): [(frame_paths, label), ...]}
            
            for cls in self.classes:
                cls_dir = self.root / cls
                for video_dir in sorted([d for d in cls_dir.iterdir() if d.is_dir()]):
                    frame_paths = sorted([p for p in video_dir.iterdir() 
                                        if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}])
                    if frame_paths:
                        # Extract base video name (remove trailing _N)
                        base_name = self._base_video_name(video_dir.name)
                        key = (cls, base_name)
                        if key not in grouped_samples:
                            grouped_samples[key] = []
                        grouped_samples[key].append((frame_paths, self.class_to_idx[cls]))
            
            # Grouped split to avoid data leakage
            group_keys = list(grouped_samples.keys())
            rng = np.random.RandomState(seed)
            indices = np.arange(len(group_keys))
            rng.shuffle(indices)
            split_point = int(len(indices) * (1 - val_ratio))
            
            if is_train:
                selected_groups = indices[:split_point]
            else:
                selected_groups = indices[split_point:]
            
            # Collect samples from selected groups
            self.samples = []
            for idx in selected_groups:
                self.samples.extend(grouped_samples[group_keys[int(idx)]])
        else:
            # Use provided samples (for train/val split)
            self.samples = samples
    
    @staticmethod
    def _base_video_name(name):
        """Remove trailing _N from video name for grouping."""
        match = re.match(r"(.+)_\d+$", name)
        return match.group(1) if match else name
    
    def __len__(self):
        return len(self.samples)
    
    def _select_indices(self, total):
        if total <= 0:
            raise ValueError("No frames")
        if total == 1:
            return torch.zeros(self.num_frames, dtype=torch.long)
        steps = max(self.num_frames * self.frame_stride, self.num_frames)
        grid = torch.linspace(0, total - 1, steps=steps)
        idxs = grid[::self.frame_stride].long()
        if idxs.numel() < self.num_frames:
            pad = idxs.new_full((self.num_frames - idxs.numel(),), idxs[-1].item())
            idxs = torch.cat([idxs, pad], dim=0)
        return idxs[:self.num_frames]
    
    def __getitem__(self, idx):
        frame_paths, label = self.samples[idx]
        total = len(frame_paths)
        idxs = self._select_indices(total)
        frames = []
        for i in idxs:
            path = frame_paths[int(i.item())]
            with Image.open(path) as img:
                img = img.convert("RGB")
                frames.append(self.to_tensor(img))
        video = torch.stack(frames)
        video = self.transform(video)
        return video, label

print("Dataset classes defined")


## 5. Load Datasets


In [ ]:
print("Loading datasets with validation split...")
train_dataset = VideoDataset(DATA_ROOT, num_frames=NUM_FRAMES, frame_stride=FRAME_STRIDE, 
                              image_size=IMG_SIZE, is_train=True, val_ratio=VAL_RATIO, seed=42)
val_dataset = VideoDataset(DATA_ROOT, num_frames=NUM_FRAMES, frame_stride=FRAME_STRIDE,
                           image_size=IMG_SIZE, is_train=False, val_ratio=VAL_RATIO, seed=42)

# Note: num_workers=0 to avoid multiprocessing issues in Jupyter notebooks on Windows
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print(f"Classes: {len(train_dataset.classes)}")
print(f"Class names: {train_dataset.classes[:10]}...")


## 6. Create Model


In [ ]:
print("Creating SOTA model...")
model = SOTAViTForAction(num_classes=len(train_dataset.classes), 
                        pretrained_name=PRETRAINED_NAME,
                        use_adapters=USE_ADAPTERS).to(DEVICE)

# Count trainable parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Trainable ratio: {trainable_params/total_params*100:.2f}%")
print(f"Model size: {total_params * 4 / 1024 / 1024:.2f} MB")  # Approximate


## 7. Training Setup


In [ ]:
# Separate parameters for different learning rates
backbone_params = []
adapter_params = []
head_params = []

for name, param in model.named_parameters():
    if not param.requires_grad:
        continue
    if 'head' in name:
        head_params.append(param)
    elif 'adapter' in name or 'temporal_attention' in name:
        adapter_params.append(param)
    else:
        backbone_params.append(param)

# Create optimizer with different learning rates
param_groups = []
if backbone_params:
    param_groups.append({"params": backbone_params, "lr": BASE_LR})
if adapter_params:
    param_groups.append({"params": adapter_params, "lr": HEAD_LR})
if head_params:
    param_groups.append({"params": head_params, "lr": HEAD_LR})

optimizer = torch.optim.AdamW(param_groups, weight_decay=WEIGHT_DECAY)

# Learning rate scheduler with warmup
def get_lr_scheduler(optimizer, num_epochs, warmup_epochs=5):
    """Create learning rate scheduler with warmup."""
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            # Warmup: linear increase
            return (epoch + 1) / warmup_epochs
        else:
            # Cosine annealing
            progress = (epoch - warmup_epochs) / (num_epochs - warmup_epochs)
            return 0.5 * (1 + np.cos(np.pi * progress))
    
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

scheduler = get_lr_scheduler(optimizer, EPOCHS, WARMUP_EPOCHS)
scaler = torch.amp.GradScaler(enabled=torch.cuda.is_available())

print(f"Optimizer: AdamW")
print(f"  Backbone LR: {BASE_LR} (params: {len(backbone_params)})")
print(f"  Adapter/Head LR: {HEAD_LR} (params: {len(adapter_params) + len(head_params)})")
print(f"  Scheduler: Warmup ({WARMUP_EPOCHS} epochs) + CosineAnnealing")


## 8. Training Functions


In [ ]:
def train_one_epoch(model, loader, optimizer, scaler, device, grad_accum_steps=1, 
                    use_mixup=True, mixup_alpha=0.4, label_smoothing=0.0):
    """Training function with Mixup and label smoothing."""
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    optimizer.zero_grad()
    
    # Criterion with label smoothing
    if label_smoothing > 0:
        criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    else:
        criterion = nn.CrossEntropyLoss()
    
    progress = tqdm(loader, desc="Train", leave=False)
    for batch_idx, (videos, labels) in enumerate(progress):
        videos = videos.to(device)
        labels = labels.to(device)
        
        # Apply Mixup
        if use_mixup and random.random() < 0.5:
            mixed_videos, labels_a, labels_b, lam = mixup_data(videos, labels, mixup_alpha)
            with torch.amp.autocast(device_type='cuda', enabled=(device.type == 'cuda')):
                logits = model(mixed_videos)
                loss = mixup_criterion(criterion, logits, labels_a, labels_b, lam)
            # For accuracy calculation, use original labels
            preds = logits.argmax(dim=1)
            correct += (lam * (preds == labels_a).float() + 
                       (1 - lam) * (preds == labels_b).float()).sum().item()
        else:
            with torch.amp.autocast(device_type='cuda', enabled=(device.type == 'cuda')):
                logits = model(videos)
                loss = criterion(logits, labels)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
        
        total += labels.size(0)
        loss_value = loss.item()
        loss = loss / grad_accum_steps
        scaler.scale(loss).backward()
        
        should_step = ((batch_idx + 1) % grad_accum_steps == 0) or (batch_idx + 1 == len(loader))
        if should_step:
            # Gradient clipping - only unscale when we're about to step
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        
        batch_size = videos.size(0)
        total_loss += loss_value * batch_size
        progress.set_postfix(loss=f"{loss_value:.4f}", acc=f"{correct / max(total, 1):.4f}")
    
    avg_loss = total_loss / max(total, 1)
    avg_acc = correct / max(total, 1)
    return avg_loss, avg_acc


def evaluate(model, loader, device, label_smoothing=0.0):
    """Evaluation function."""
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    if label_smoothing > 0:
        criterion = nn.CrossEntropyLoss(label_smoothing=label_smoothing)
    else:
        criterion = nn.CrossEntropyLoss()
    
    with torch.no_grad():
        progress = tqdm(loader, desc="Val", leave=False)
        for videos, labels in progress:
            videos = videos.to(device)
            labels = labels.to(device)
            
            with torch.amp.autocast(device_type='cuda', enabled=(device.type == 'cuda')):
                logits = model(videos)
                loss = criterion(logits, labels)
            
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            total_loss += loss.item() * labels.size(0)
            
            progress.set_postfix(loss=f"{loss.item():.4f}", acc=f"{correct / max(total, 1):.4f}")
    
    avg_loss = total_loss / max(total, 1)
    avg_acc = correct / max(total, 1)
    return avg_loss, avg_acc

print("Training functions defined (with Mixup, label smoothing, gradient clipping)")


## 9. Training Loop


In [ ]:
best_val_acc = 0.0
checkpoint_path = Path('./sota_vit_best.pt')
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
patience_counter = 0

print("\n" + "="*60)
print("Starting SOTA Training")
print("="*60)

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch + 1}/{EPOCHS}")
    
    # Training
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, scaler, DEVICE, GRAD_ACCUM_STEPS,
        use_mixup=True, mixup_alpha=MIXUP_ALPHA, label_smoothing=LABEL_SMOOTHING
    )
    
    # Validation
    val_loss, val_acc = evaluate(model, val_loader, DEVICE, label_smoothing=LABEL_SMOOTHING)
    
    # Update learning rate
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    print(f"  Train: Loss={train_loss:.4f}, Acc={train_acc:.4f}")
    print(f"  Val:   Loss={val_loss:.4f}, Acc={val_acc:.4f}")
    print(f"  LR: {current_lr:.6f}")
    
    # Save best model based on validation accuracy
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        patience_counter = 0
        torch.save({
            'model': model.state_dict(), 
            'classes': train_dataset.classes, 
            'val_acc': best_val_acc,
            'train_acc': train_acc,
            'epoch': epoch + 1,
            'history': history
        }, checkpoint_path)
        print(f"  ✓ Best model saved (val_acc: {best_val_acc:.4f})")
    else:
        patience_counter += 1
        print(f"  No improvement ({patience_counter}/{EARLY_STOP_PATIENCE})")
    
    # Early stopping
    if patience_counter >= EARLY_STOP_PATIENCE:
        print(f"\nEarly stopping triggered after {epoch + 1} epochs")
        break

print("\n" + "="*60)
print(f"Training completed!")
print(f"Best validation accuracy: {best_val_acc:.4f}")
print(f"Model saved to: {checkpoint_path}")
print("="*60)


## 10. Plot Results


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train', marker='o', linewidth=2)
axes[0].plot(history['val_loss'], label='Val', marker='s', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('Loss', fontsize=12)
axes[0].set_title('Training Loss', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)
axes[0].grid(alpha=0.3)

# Accuracy
axes[1].plot(history['train_acc'], label='Train', marker='o', linewidth=2)
axes[1].plot(history['val_acc'], label='Val', marker='s', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('Accuracy', fontsize=12)
axes[1].set_title('Training Accuracy', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)
axes[1].grid(alpha=0.3)
axes[1].set_ylim([0, 1])

# Fix for NumPy 2.4.0rc1 compatibility: Skip tight_layout if NumPy 2.4.x
import numpy as np
if np.__version__.startswith('2.4'):
    # NumPy 2.4.0rc1 has bug with matplotlib's tight_layout
    plt.subplots_adjust(left=0.1, right=0.95, top=0.9, bottom=0.1, wspace=0.3)
    plt.savefig('sota_vit_training.png', dpi=100, bbox_inches=None)
else:
    plt.tight_layout()
    plt.savefig('sota_vit_training.png', dpi=100, bbox_inches='tight')

plt.show()

print("Plot saved: sota_vit_training.png")


## 11. Test-Time Augmentation (TTA) for Inference


In [ ]:
def test_time_augment(video, model, device, num_crops=5, num_flips=2):
    """
    Apply Test-Time Augmentation to a single video.
    
    Args:
        video: [B, T, C, H, W] tensor
        model: model to use for inference
        device: device to run on
        num_crops: number of spatial crops (1=center, 5=center+4corners)
        num_flips: number of flips (1=original, 2=original+flipped)
    
    Returns:
        Averaged logits [num_classes]
    """
    B, T, C, H, W = video.shape
    video = video.to(device)
    
    all_logits = []
    
    # Original video
    with torch.no_grad():
        logits = model(video)
        all_logits.append(logits)
    
    # Horizontal flip
    if num_flips > 1:
        video_flipped = torch.flip(video, dims=[-1])  # Flip width dimension
        with torch.no_grad():
            logits = model(video_flipped)
            all_logits.append(logits)
    
    # Multiple crops
    if num_crops > 1:
        crop_size = int(H * 0.875)  # 87.5% of original size
        crops = [
            (0, 0),  # Top-left
            (0, W - crop_size),  # Top-right
            (H - crop_size, 0),  # Bottom-left
            (H - crop_size, W - crop_size),  # Bottom-right
            ((H - crop_size) // 2, (W - crop_size) // 2),  # Center
        ]
        
        for i, (top, left) in enumerate(crops[:num_crops-1]):  # -1 because we already have original
            video_crop = video[:, :, :, top:top+crop_size, left:left+crop_size]
            video_crop = F.interpolate(video_crop.view(B*T, C, crop_size, crop_size),
                                      size=(H, W), mode='bilinear', align_corners=False)
            video_crop = video_crop.view(B, T, C, H, W)
            
            with torch.no_grad():
                logits = model(video_crop)
                all_logits.append(logits)
    
    # Average all predictions
    all_logits = torch.stack(all_logits)
    avg_logits = all_logits.mean(dim=0)
    
    return avg_logits

print("Test-Time Augmentation function defined")
print("  Supports: 5 crops (center + 4 corners) + 2 flips (original + horizontal)")


## 12. Inference with TTA


In [ ]:
# Load best model
print("Loading best model checkpoint...")
checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
classes = checkpoint['classes']

model = SOTAViTForAction(num_classes=len(classes), pretrained_name=PRETRAINED_NAME,
                        use_adapters=USE_ADAPTERS).to(DEVICE)
model.load_state_dict(checkpoint['model'])
model.eval()
print(f"Model loaded (val_acc: {checkpoint.get('val_acc', 0):.4f})\n")

# Inference on validation set with TTA
print("Running inference with TTA on validation set...")
predictions = []
ground_truth = []
correct = 0
total = 0

with torch.no_grad():
    for videos, labels in tqdm(val_loader, desc="Inference"):
        videos = videos.to(DEVICE)
        labels = labels.to(DEVICE)
        
        # Apply TTA
        batch_logits = []
        for i in range(videos.shape[0]):
            video = videos[i:i+1]  # [1, T, C, H, W]
            logits = test_time_augment(video, model, DEVICE, num_crops=5, num_flips=2)
            batch_logits.append(logits)
        
        batch_logits = torch.cat(batch_logits, dim=0)
        preds = batch_logits.argmax(dim=1)
        
        correct += (preds == labels).sum().item()
        total += labels.size(0)
        
        predictions.extend(preds.cpu().numpy())
        ground_truth.extend(labels.cpu().numpy())

tta_acc = correct / total
print(f"\nValidation accuracy with TTA: {tta_acc:.4f}")
print(f"Validation accuracy without TTA: {checkpoint.get('val_acc', 0):.4f}")
print(f"Improvement: {tta_acc - checkpoint.get('val_acc', 0):.4f}")
